
# Stage 3 — выбор каналов, временного и частотного окна
## `open_hand` против `all_other_gestures`

Этот ноутбук не обучает финальные модели. Его задача — **зафиксированным и одинаковым способом** просмотреть все доступные каналы выбранного пациента и сформировать кандидатов для последующего ablation-эксперимента.

На каждом графике временная ось задаётся относительно **LSL-метки**:

- `0.0 s` — LSL-метка;
- `+0.1 s` — оценённое начало движения робота;
- `0.0–1.0 s` — текущее окно входа декодера;
- следовательно, текущее окно соответствует `−0.1…0.9 s` относительно начала движения робота.

### Принципы

1. Список эпох фиксируется по `study_definition`: одинаковые эпохи используются для всех каналов и всех экранов.
2. Фильтрация, CAR, baseline и временная система координат одинаковы для всех каналов.
3. `ch_to_keep[patient]` используется как полный набор кандидатов для просмотра.
4. `best_ch_by_power[patient]` показывается только как **предыдущий выбор**, а не как правильный ответ.
5. Карты Cohen's d и AUC используются для скрининга сигнала. Это **не итоговая accuracy модели**.
6. Окончательная проверка selected/control/time/frequency выполняется следующим отдельным ablation-runner с фиксированными folds.

### Структура выходных папок

- `00_run_info` — конфигурация запуска, списки каналов, памятка;
- `01_inventory_and_qc` — состав данных, шум и амплитудные выбросы;
- `02_erp_channel_screening` — ERP и trial heatmaps по всем каналам;
- `03_tfr_channel_screening` — средние TFR `open`, `other`, `difference`;
- `04_discriminability_maps` — Cohen's d и `|AUC−0.5|`;
- `05_channel_ranking` — таблицы и стабильность каналов между сессиями;
- `06_time_window_screening` — сравнение временных окон;
- `07_frequency_band_screening` — сравнение частотных диапазонов;
- `08_selected_vs_control` — предыдущий selected-набор и кандидаты в control;
- `09_final_review` — сводная таблица, которую можно отредактировать вручную.


## 0. Параметры запуска

In [1]:

from pathlib import Path

# =========================== ОСНОВНОЙ ВЫБОР ===========================
PATIENT_ID = "s11"

# Конфиг, присланный вместе с ноутбуком.
CONFIG_PATH = Path("/trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/config.py")

# Архив/папка, созданная prepare_study_definition.py.
STUDY_DEF_PATH = Path('/trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/study_definition_open_vs_all')

# Корень с исходными EDF. Используется, если путь из manifest недоступен.
DATA_ROOT = Path("/trinity/home/t.samsonov/notebooks/Pirogov/PirogovDATA")

# Все изображения и CSV попадут сюда.
OUTPUT_STORAGE = Path("/trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/stage3_input_screening")
OUTPUT_ROOT = OUTPUT_STORAGE / PATIENT_ID

# None -> все доступные сессии пациента.
# Пример: ["2025-09-02_s11/session_1"]
SESSIONS_TO_USE = None

# =========================== ВРЕМЯ ===========================
ROBOT_DELAY_S = 0.100
ROBOT_ONSET_FROM_MARKER_S = ROBOT_DELAY_S

# Общий интервал просмотра относительно LSL-метки.
# Он соответствует примерно -1...+2 s относительно движения робота.
EPOCH_TMIN_MARKER = -0.9
EPOCH_TMAX_MARKER = 2.1

# Pipeline baseline относительно LSL-метки.
PIPELINE_BASELINE_MARKER = (-0.1, 0.0)

# Текущее окно декодера относительно LSL-метки.
CURRENT_TIME_WINDOW = (0.0, 1.0)
CURRENT_FREQ_RANGE = (0.1, 59.4)

TIME_WINDOWS = {
    "T0_current_marker_0_1": (0.0, 1.0),
    "T1_robot_0_1": (0.1, 1.1),
    "T2_robot_0_0p5": (0.1, 0.6),
    "T3_robot_0_1p5": (0.1, 1.6),
    "T4_robot_0_2": (0.1, 2.1),
}

FREQUENCY_BANDS = {
    "F0_current_0p1_59p4": (0.1, 59.4),
    "F1_low_0p1_30": (0.1, 30.0),
    "F2_alpha_beta_8_30": (8.0, 30.0),
    "F3_low_gamma_30_59p4": (30.0, 59.4),
    "F4_full_0p1_120": (0.1, 120.0),
    "F5_upper_59p4_120": (59.4, 120.0),
}

# =========================== PREPROCESSING ===========================
NOTCH_FREQS = [50.0, 100.0, 150.0]
L_FREQ = 0.1
H_FREQ = 120.0
APPLY_CAR = True

# Использовать именно accepted_after_rejection из manifest.
# Это гарантирует один и тот же набор эпох для всех каналов.
USE_MANIFEST_ACCEPTED_EPOCHS = True

# =========================== TFR / MAPS ===========================
TFR_FREQS = 80
TFR_DECIM = 10
N_JOBS = 1
COMPUTE_AUC = True
AUC_CHUNK_SIZE = 4000

# Ограничение числа trial-строк только на изображении.
# В расчётах используются все эпохи.
MAX_TRIALS_PER_CLASS_ON_FIGURE = 180

# =========================== ЭТАПЫ ===========================
RUN_QC = True
RUN_ERP = True
RUN_TFR = True
RUN_DISCRIMINABILITY = True
RUN_SESSION_STABILITY = True
RUN_TIME_SCREENING = True
RUN_FREQUENCY_SCREENING = True
RUN_SELECTED_CONTROL_REVIEW = True

# Опционально: после просмотра можно задать окончательные наборы вручную
# и повторно выполнить последние блоки.
MANUAL_SELECTED_CHANNELS = None
MANUAL_CONTROL_CHANNELS = None

# =========================== FIGURES ===========================
FIG_DPI = 150
CHANNELS_PER_PAGE = 6
INTERPOLATION = "nearest"
SAVE_SVG = False
RANDOM_SEED = 42


## 1. Импорты, конфиг и служебные функции

In [2]:

import io
import json
import math
import zipfile
import importlib.util
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from scipy.stats import rankdata
import mne
from IPython.display import display, Markdown, Image

plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI


def load_python_config(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Config not found: {path}")
    spec = importlib.util.spec_from_file_location("user_config", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


cfg = load_python_config(CONFIG_PATH)

if PATIENT_ID not in cfg.ch_to_keep:
    raise KeyError(f"{PATIENT_ID} отсутствует в ch_to_keep")

CONFIG_CANDIDATE_CHANNELS = list(cfg.ch_to_keep[PATIENT_ID])
CONFIG_PREVIOUS_BEST = list(cfg.best_ch_by_power.get(PATIENT_ID, []))
EPOCH_REJECT_THRESHOLD_V = float(
    cfg.epoch_thresh_dict.get(
        PATIENT_ID,
        cfg.epoch_thresh_dict.get("default", np.nan),
    )
)

print("Patient:", PATIENT_ID)
print("Candidate channels from ch_to_keep:", CONFIG_CANDIDATE_CHANNELS)
print("Previous best_ch_by_power:", CONFIG_PREVIOUS_BEST)
print("Epoch threshold from config, V:", EPOCH_REJECT_THRESHOLD_V)


Patient: s11
Candidate channels from ch_to_keep: ['Fp1', 'F7', 'Ft7', 'T3', 'F8', 'Fp2', 'Fpz', 'F3', 'Fc3', 'Tp7', 'Fc4']
Previous best_ch_by_power: ['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']
Epoch threshold from config, V: 0.0007


In [3]:

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


DIRS = {
    "run": ensure_dir(OUTPUT_ROOT / "00_run_info"),
    "qc": ensure_dir(OUTPUT_ROOT / "01_inventory_and_qc"),
    "erp": ensure_dir(OUTPUT_ROOT / "02_erp_channel_screening"),
    "erp_cards": ensure_dir(OUTPUT_ROOT / "02_erp_channel_screening" / "per_channel_cards"),
    "tfr": ensure_dir(OUTPUT_ROOT / "03_tfr_channel_screening"),
    "tfr_cards": ensure_dir(OUTPUT_ROOT / "03_tfr_channel_screening" / "per_channel_cards"),
    "disc": ensure_dir(OUTPUT_ROOT / "04_discriminability_maps"),
    "disc_cards": ensure_dir(OUTPUT_ROOT / "04_discriminability_maps" / "per_channel_cards"),
    "ranking": ensure_dir(OUTPUT_ROOT / "05_channel_ranking"),
    "time": ensure_dir(OUTPUT_ROOT / "06_time_window_screening"),
    "freq": ensure_dir(OUTPUT_ROOT / "07_frequency_band_screening"),
    "control": ensure_dir(OUTPUT_ROOT / "08_selected_vs_control"),
    "final": ensure_dir(OUTPUT_ROOT / "09_final_review"),
    "cache": ensure_dir(OUTPUT_ROOT / "_cache"),
}


README = f"""
STAGE 3 INPUT SCREENING — {PATIENT_ID}

00_run_info
  Проверить конфигурацию запуска и полный список каналов.

01_inventory_and_qc
  Сначала проверить отсутствующие каналы, число эпох, peak-to-peak,
  baseline RMS и долю эпох выше patient-specific threshold.
  Эти показатели не выбирают информативный канал, а исключают явно плохие.

02_erp_channel_screening
  Смотреть форму вызванного ответа, стабильность знака и trial heatmaps.
  Хороший средний ERP, созданный несколькими выбросами, здесь будет заметен.

03_tfr_channel_screening
  Смотреть open_hand, all_other и open-other с общей шкалой внутри канала.

04_discriminability_maps
  Основные количественные карты: Cohen's d и |AUC-0.5|.
  Это signal-level screening, а не accuracy классификатора.

05_channel_ranking
  Сводка показателей и стабильность между сессиями.
  Предыдущий best_ch_by_power только подсвечивается, но не навязывается.

06_time_window_screening
  Сравнение заранее зафиксированных T0-T4.

07_frequency_band_screening
  Сравнение заранее зафиксированных F0-F5.

08_selected_vs_control
  Формирование равных по размеру selected/control подмножеств.
  При недостатке оставшихся каналов размер автоматически уменьшается.

09_final_review
  CSV-шаблон для ручного решения. После его заполнения нужен отдельный
  fixed-fold SVM ablation-runner.
""".strip()

(DIRS["run"] / "README_WHAT_TO_LOOK_AT.txt").write_text(README, encoding="utf-8")
print(README)


STAGE 3 INPUT SCREENING — s11

00_run_info
  Проверить конфигурацию запуска и полный список каналов.

01_inventory_and_qc
  Сначала проверить отсутствующие каналы, число эпох, peak-to-peak,
  baseline RMS и долю эпох выше patient-specific threshold.
  Эти показатели не выбирают информативный канал, а исключают явно плохие.

02_erp_channel_screening
  Смотреть форму вызванного ответа, стабильность знака и trial heatmaps.
  Хороший средний ERP, созданный несколькими выбросами, здесь будет заметен.

03_tfr_channel_screening
  Смотреть open_hand, all_other и open-other с общей шкалой внутри канала.

04_discriminability_maps
  Основные количественные карты: Cohen's d и |AUC-0.5|.
  Это signal-level screening, а не accuracy классификатора.

05_channel_ranking
  Сводка показателей и стабильность между сессиями.
  Предыдущий best_ch_by_power только подсвечивается, но не навязывается.

06_time_window_screening
  Сравнение заранее зафиксированных T0-T4.

07_frequency_band_screening
  Сравнение з

In [4]:

def load_table(study_def_path: Path, relative_name: str) -> pd.DataFrame:
    if study_def_path.is_dir():
        return pd.read_csv(study_def_path / relative_name)

    if study_def_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(study_def_path, "r") as zf:
            target = next(
                (
                    name for name in zf.namelist()
                    if name == relative_name or name.endswith("/" + relative_name)
                ),
                None,
            )
            if target is None:
                raise FileNotFoundError(relative_name)
            with zf.open(target) as f:
                return pd.read_csv(f)

    raise ValueError(f"Unsupported study definition: {study_def_path}")


def parse_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().map({"true": True, "false": False})


def resolve_edf_path(row: pd.Series) -> Path:
    original = Path(str(row["edf_path"]))
    if original.exists():
        return original

    derived = DATA_ROOT / str(row["session"]) / original.name
    if derived.exists():
        return derived

    session_dir = DATA_ROOT / str(row["session"])
    matches = sorted(session_dir.glob("*.edf")) if session_dir.exists() else []
    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"EDF not found. Manifest path: {original}; derived path: {derived}"
    )


def resolve_and_rename_channels(raw: mne.io.BaseRaw, requested: list[str]):
    lower_map = {}
    for actual in raw.ch_names:
        lower_map.setdefault(actual.lower(), []).append(actual)

    rename_map = {}
    missing = []
    ordered_actual = []

    for canonical in requested:
        matches = lower_map.get(canonical.lower(), [])
        if len(matches) == 1:
            actual = matches[0]
            ordered_actual.append(actual)
            if actual != canonical:
                rename_map[actual] = canonical
        elif len(matches) == 0:
            missing.append(canonical)
        else:
            raise RuntimeError(
                f"Ambiguous case-insensitive channel name {canonical}: {matches}"
            )

    if not ordered_actual:
        raise RuntimeError("None of the configured channels were found in EDF")

    out = raw.copy().pick_channels(ordered_actual, ordered=True)
    if rename_map:
        out.rename_channels(rename_map)

    available_canonical = [ch for ch in requested if ch in out.ch_names]
    out.reorder_channels(available_canonical)
    return out, available_canonical, missing


def preprocess_candidate_channels(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    picks = np.arange(len(raw.ch_names))
    valid_notches = [f for f in NOTCH_FREQS if f < raw.info["sfreq"] / 2]

    if valid_notches:
        raw.notch_filter(valid_notches, picks=picks, verbose="ERROR")
    raw.filter(L_FREQ, H_FREQ, picks=picks, verbose="ERROR")

    if APPLY_CAR:
        data = raw.get_data()
        data = data - data.mean(axis=0, keepdims=True)
        raw._data[:] = data

    return raw


def save_figure(fig, path_without_suffix: Path):
    ensure_dir(path_without_suffix.parent)
    fig.savefig(path_without_suffix.with_suffix(".png"), bbox_inches="tight")
    if SAVE_SVG:
        fig.savefig(path_without_suffix.with_suffix(".svg"), bbox_inches="tight")
    plt.close(fig)


def safe_name(value: str) -> str:
    return (
        str(value)
        .replace("/", "__")
        .replace(" ", "_")
        .replace(":", "_")
    )


def timing_handles(include_window=True):
    handles = [
        Line2D([0], [0], linestyle="--", linewidth=1.2, label="LSL marker: 0 s"),
        Line2D(
            [0], [0],
            linestyle="-.",
            linewidth=1.2,
            label=f"Robot onset: +{ROBOT_ONSET_FROM_MARKER_S:.1f} s",
        ),
    ]
    if include_window:
        handles.append(
            Patch(
                alpha=0.12,
                label=(
                    f"Current decoder window: "
                    f"{CURRENT_TIME_WINDOW[0]:.1f}–{CURRENT_TIME_WINDOW[1]:.1f} s"
                ),
            )
        )
    return handles


def add_timing_annotations(ax, shade_current_window=False):
    ax.axvline(0.0, linestyle="--", linewidth=1.2, zorder=10)
    ax.axvline(
        ROBOT_ONSET_FROM_MARKER_S,
        linestyle="-.",
        linewidth=1.2,
        alpha=0.8,
        zorder=10,
    )
    if shade_current_window:
        ax.axvspan(
            CURRENT_TIME_WINDOW[0],
            CURRENT_TIME_WINDOW[1],
            alpha=0.08,
            zorder=0,
        )


def add_decoder_rectangle(ax):
    rect = Rectangle(
        (CURRENT_TIME_WINDOW[0], CURRENT_FREQ_RANGE[0]),
        CURRENT_TIME_WINDOW[1] - CURRENT_TIME_WINDOW[0],
        CURRENT_FREQ_RANGE[1] - CURRENT_FREQ_RANGE[0],
        fill=False,
        linewidth=1.3,
        linestyle="--",
        zorder=11,
    )
    ax.add_patch(rect)


def timing_caption():
    return (
        "LSL marker = 0 s; robot onset = +0.1 s; "
        "current decoder input = 0–1 s from marker "
        "(−0.1…0.9 s from robot onset)"
    )


def robust_symmetric_limit(arrays, q=0.995, fallback=1.0):
    vals = []
    for arr in arrays:
        x = np.asarray(arr)
        x = np.abs(x[np.isfinite(x)])
        if x.size:
            vals.append(x)
    if not vals:
        return fallback
    return float(np.quantile(np.concatenate(vals), q))


def power_to_db(power):
    eps = np.finfo(float).tiny
    return 10.0 * np.log10(np.maximum(power, eps))


## 2. Загрузка manifest и всех кандидатных каналов

In [5]:

manifest = load_table(STUDY_DEF_PATH, "sample_manifest.csv")
manifest["included_in_main_task"] = parse_bool(manifest["included_in_main_task"])
manifest["accepted_after_rejection"] = parse_bool(manifest["accepted_after_rejection"])

patient_manifest = manifest[manifest["subject"] == PATIENT_ID].copy()

if SESSIONS_TO_USE is not None:
    patient_manifest = patient_manifest[
        patient_manifest["session"].isin(SESSIONS_TO_USE)
    ].copy()

patient_manifest = patient_manifest[
    patient_manifest["included_in_main_task"] == True
].copy()

if USE_MANIFEST_ACCEPTED_EPOCHS:
    patient_manifest = patient_manifest[
        patient_manifest["accepted_after_rejection"] == True
    ].copy()

if patient_manifest.empty:
    raise RuntimeError(f"No task events found for {PATIENT_ID}")

patient_sessions = sorted(patient_manifest["session"].unique())
print("Sessions:", patient_sessions)
print("Task rows:", len(patient_manifest))


Sessions: ['2025-09-05_s11/session_1', '2025-09-05_s11/session_2']
Task rows: 535


In [6]:

def build_session_epochs(session_df: pd.DataFrame):
    session_df = session_df.sort_values(
        ["sample", "annotation_index"]
    ).reset_index(drop=True)

    edf_path = resolve_edf_path(session_df.iloc[0])
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")

    raw, available_channels, missing_channels = resolve_and_rename_channels(
        raw,
        CONFIG_CANDIDATE_CHANNELS,
    )
    raw = preprocess_candidate_channels(raw)

    rows = session_df[
        session_df["event_code"].isin([1,2,3,4,5,6,7,8,9,10])
    ].copy().reset_index(drop=True)

    events = np.zeros((len(rows), 3), dtype=int)
    events[:, 0] = rows["sample"].astype(int).to_numpy()
    events[:, 2] = rows["event_code"].astype(int).to_numpy()

    metadata = rows[[
        "subject",
        "session",
        "epoch_id",
        "gesture_name",
        "event_code",
        "class_name",
        "marker_onset_s",
        "robot_onset_s",
    ]].copy()

    event_id = {
        f"gesture_{code}": code
        for code in sorted(rows["event_code"].unique())
    }

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=EPOCH_TMIN_MARKER,
        tmax=EPOCH_TMAX_MARKER,
        baseline=None,
        preload=True,
        metadata=metadata,
        event_repeated="drop",
        on_missing="ignore",
        verbose="ERROR",
    )

    return {
        "session": rows["session"].iloc[0],
        "edf_path": edf_path,
        "epochs": epochs,
        "available_channels": available_channels,
        "missing_channels": missing_channels,
        "n_requested_events": len(rows),
        "n_created_epochs": len(epochs),
    }


session_objects = []
for session_name in patient_sessions:
    session_df = patient_manifest[
        patient_manifest["session"] == session_name
    ].copy()
    obj = build_session_epochs(session_df)
    session_objects.append(obj)
    print(
        f"{session_name}: {obj['n_created_epochs']}/{obj['n_requested_events']} epochs; "
        f"channels={obj['available_channels']}; missing={obj['missing_channels']}"
    )

channel_orders = [tuple(obj["epochs"].ch_names) for obj in session_objects]
if len(set(channel_orders)) != 1:
    raise RuntimeError(
        "The available channel list differs between sessions. "
        "Inspect 01_inventory_and_qc before concatenation."
    )

patient_epochs = mne.concatenate_epochs(
    [obj["epochs"] for obj in session_objects],
    add_offset=True,
)

patient_epochs_bc = patient_epochs.copy().apply_baseline(
    PIPELINE_BASELINE_MARKER
)

AVAILABLE_CHANNELS = list(patient_epochs.ch_names)
PREVIOUS_BEST_AVAILABLE = [
    ch for ch in CONFIG_PREVIOUS_BEST if ch in AVAILABLE_CHANNELS
]
REMAINING_AVAILABLE = [
    ch for ch in AVAILABLE_CHANNELS if ch not in PREVIOUS_BEST_AVAILABLE
]

print("\nPatient-level epochs:", len(patient_epochs))
print("Available candidate channels:", AVAILABLE_CHANNELS)
print("Previous best available:", PREVIOUS_BEST_AVAILABLE)
print("Remaining candidates:", REMAINING_AVAILABLE)


2025-09-05_s11/session_1: 135/135 epochs; channels=['Fp1', 'F7', 'Ft7', 'T3', 'F8', 'Fp2', 'Fpz', 'F3', 'Fc3', 'Tp7', 'Fc4']; missing=[]
2025-09-05_s11/session_2: 400/400 epochs; channels=['Fp1', 'F7', 'Ft7', 'T3', 'F8', 'Fp2', 'Fpz', 'F3', 'Fc3', 'Tp7', 'Fc4']; missing=[]
Adding metadata with 8 columns
535 matching events found
No baseline correction applied


/tmp/ipykernel_2596184/2961437504.py:83: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  patient_epochs = mne.concatenate_epochs(


Applying baseline correction (mode: mean)

Patient-level epochs: 535
Available candidate channels: ['Fp1', 'F7', 'Ft7', 'T3', 'F8', 'Fp2', 'Fpz', 'F3', 'Fc3', 'Tp7', 'Fc4']
Previous best available: ['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']
Remaining candidates: ['Ft7', 'T3', 'Fc3', 'Fc4']


In [7]:

run_config = {
    "patient_id": PATIENT_ID,
    "config_path": str(CONFIG_PATH),
    "study_definition": str(STUDY_DEF_PATH),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "sessions": patient_sessions,
    "candidate_channels_from_config": CONFIG_CANDIDATE_CHANNELS,
    "available_channels": AVAILABLE_CHANNELS,
    "previous_best_available": PREVIOUS_BEST_AVAILABLE,
    "epoch_reject_threshold_v": EPOCH_REJECT_THRESHOLD_V,
    "use_manifest_accepted_epochs": USE_MANIFEST_ACCEPTED_EPOCHS,
    "epoch_window_marker_s": [EPOCH_TMIN_MARKER, EPOCH_TMAX_MARKER],
    "baseline_marker_s": list(PIPELINE_BASELINE_MARKER),
    "current_time_window_marker_s": list(CURRENT_TIME_WINDOW),
    "current_frequency_range_hz": list(CURRENT_FREQ_RANGE),
    "time_windows": TIME_WINDOWS,
    "frequency_bands": FREQUENCY_BANDS,
}

(DIRS["run"] / "run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

channel_inventory = pd.DataFrame({
    "channel": CONFIG_CANDIDATE_CHANNELS,
    "available_in_all_selected_sessions": [
        ch in AVAILABLE_CHANNELS for ch in CONFIG_CANDIDATE_CHANNELS
    ],
    "previous_best_ch_by_power": [
        ch in CONFIG_PREVIOUS_BEST for ch in CONFIG_CANDIDATE_CHANNELS
    ],
})
channel_inventory.to_csv(
    DIRS["run"] / "channel_inventory.csv",
    index=False,
)
display(channel_inventory)


,channel,available_in_all_selected_sessions,previous_best_ch_by_power
0,Fp1,True,True
1,F7,True,True
2,Ft7,True,False
3,T3,True,False
4,F8,True,True
5,Fp2,True,True
6,Fpz,True,True
7,F3,True,True
8,Fc3,True,False
9,Tp7,True,True


## 3. Этап A — inventory и QC

In [8]:

def compute_qc_table(epochs: mne.Epochs) -> pd.DataFrame:
    data = epochs.get_data()  # epochs × channels × time
    times = epochs.times
    baseline_mask = (
        (times >= PIPELINE_BASELINE_MARKER[0])
        & (times <= PIPELINE_BASELINE_MARKER[1])
    )

    ptp = np.ptp(data, axis=-1)
    baseline_rms = np.sqrt(np.mean(data[:, :, baseline_mask] ** 2, axis=-1))

    rows = []
    for ch_idx, ch_name in enumerate(epochs.ch_names):
        channel_ptp = ptp[:, ch_idx]
        channel_rms = baseline_rms[:, ch_idx]
        rows.append({
            "channel": ch_name,
            "previous_best": ch_name in PREVIOUS_BEST_AVAILABLE,
            "median_peak_to_peak_v": float(np.median(channel_ptp)),
            "p95_peak_to_peak_v": float(np.quantile(channel_ptp, 0.95)),
            "max_peak_to_peak_v": float(np.max(channel_ptp)),
            "median_baseline_rms_v": float(np.median(channel_rms)),
            "fraction_epochs_ptp_above_config_threshold": float(
                np.mean(channel_ptp > EPOCH_REJECT_THRESHOLD_V)
            ),
        })
    return pd.DataFrame(rows)


if RUN_QC:
    qc_df = compute_qc_table(patient_epochs)
    qc_df.to_csv(DIRS["qc"] / "channel_qc_summary.csv", index=False)
    display(qc_df)

    metrics = [
        ("median_peak_to_peak_v", "Median peak-to-peak, V"),
        ("median_baseline_rms_v", "Median baseline RMS, V"),
        (
            "fraction_epochs_ptp_above_config_threshold",
            "Fraction above config threshold",
        ),
    ]

    fig, axes = plt.subplots(
        len(metrics), 1,
        figsize=(12, 3.4 * len(metrics)),
        constrained_layout=True,
    )
    for ax, (column, title) in zip(axes, metrics):
        ordered = qc_df.sort_values(column, ascending=False)
        ax.bar(ordered["channel"], ordered[column])
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=45)
        for x, (_, row) in enumerate(ordered.iterrows()):
            if row["previous_best"]:
                ax.text(x, row[column], "previous best", rotation=90, va="bottom", fontsize=7)

    fig.suptitle(f"{PATIENT_ID} | channel QC | fixed accepted epoch set")
    save_figure(fig, DIRS["qc"] / "01_channel_qc_metrics")

    class_counts = (
        patient_epochs.metadata
        .groupby(["session", "class_name"])
        .size()
        .unstack(fill_value=0)
    )
    class_counts.to_csv(DIRS["qc"] / "epoch_counts_by_session_and_class.csv")
    display(class_counts)


,channel,previous_best,median_peak_to_peak_v,p95_peak_to_peak_v,max_peak_to_peak_v,median_baseline_rms_v,fraction_epochs_ptp_above_config_threshold
0,Fp1,True,0.000211,0.000280,0.000590,0.000028,0.000000
1,F7,True,0.000286,0.000394,0.000819,0.000042,0.001869
2,Ft7,False,0.000117,0.000166,0.000756,0.000018,0.001869
3,T3,False,0.000117,0.000155,0.000710,0.000017,0.001869
4,F8,True,0.000167,0.000210,0.000593,0.000026,0.000000
5,Fp2,True,0.000081,0.000115,0.000768,0.000012,0.001869
6,Fpz,True,0.000113,0.000143,0.001132,0.000017,0.001869
7,F3,True,0.000075,0.000106,0.000557,0.000012,0.000000
8,Fc3,False,0.000094,0.000127,0.001001,0.000014,0.001869
9,Tp7,True,0.000329,0.000478,0.001289,0.000053,0.001869


class_name,all_other_gestures,open_hand
session,,
2025-09-05_s11/session_1,67,68
2025-09-05_s11/session_2,200,200



### Как читать QC

- Большая амплитуда сама по себе не означает информативность.
- Канал с хорошей class difference, но с большим количеством единичных выбросов, нельзя выбирать только по среднему ERP.
- `fraction_epochs_ptp_above_config_threshold` здесь является **аудитом на фиксированном наборе эпох**. Ноутбук не удаляет разные эпохи для разных каналов.


## 4. Этап B — ERP screening по всем каналам

In [9]:

def binary_data(epochs: mne.Epochs):
    labels = (epochs.events[:, 2] == 9).astype(int)
    data = epochs.get_data()
    return data[labels == 1], data[labels == 0], labels


def sem(x, axis=0):
    return np.std(x, axis=axis, ddof=1) / np.sqrt(max(x.shape[axis], 1))


def plot_erp_overlay_pages(epochs: mne.Epochs, out_dir: Path):
    open_data, other_data, _ = binary_data(epochs)
    times = epochs.times

    open_mean = open_data.mean(axis=0)
    other_mean = other_data.mean(axis=0)
    open_sem = sem(open_data, axis=0)
    other_sem = sem(other_data, axis=0)

    n_pages = math.ceil(len(epochs.ch_names) / CHANNELS_PER_PAGE)

    for page in range(n_pages):
        start = page * CHANNELS_PER_PAGE
        stop = min((page + 1) * CHANNELS_PER_PAGE, len(epochs.ch_names))
        idxs = list(range(start, stop))

        fig, axes = plt.subplots(
            len(idxs), 1,
            figsize=(14, max(3.2, 2.6 * len(idxs))),
            sharex=True,
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)

        for ax, ch_idx in zip(axes, idxs):
            line_open = ax.plot(times, open_mean[ch_idx], label="open_hand mean")[0]
            ax.fill_between(
                times,
                open_mean[ch_idx] - open_sem[ch_idx],
                open_mean[ch_idx] + open_sem[ch_idx],
                alpha=0.18,
            )
            line_other = ax.plot(times, other_mean[ch_idx], label="all_other mean")[0]
            ax.fill_between(
                times,
                other_mean[ch_idx] - other_sem[ch_idx],
                other_mean[ch_idx] + other_sem[ch_idx],
                alpha=0.18,
            )
            add_timing_annotations(ax, shade_current_window=True)
            ax.set_title(
                f"{epochs.ch_names[ch_idx]}"
                + (" | previous best" if epochs.ch_names[ch_idx] in PREVIOUS_BEST_AVAILABLE else "")
            )
            ax.set_ylabel("Amplitude")

        axes[-1].set_xlabel("Time from LSL marker, s")
        class_handles = [line_open, line_other]
        axes[0].legend(
            handles=class_handles + timing_handles(True),
            loc="upper right",
            framealpha=0.9,
        )
        fig.suptitle(
            f"{PATIENT_ID} | ERP means ± SEM | page {page+1}/{n_pages}\n"
            + timing_caption()
        )
        save_figure(fig, out_dir / f"01_erp_overlay_page_{page+1:02d}")


def plot_erp_difference_heatmap(epochs: mne.Epochs, out_path: Path):
    open_data, other_data, _ = binary_data(epochs)
    diff = open_data.mean(axis=0) - other_data.mean(axis=0)
    v = robust_symmetric_limit([diff])

    fig, ax = plt.subplots(
        figsize=(14, max(5, 0.45 * len(epochs.ch_names))),
        constrained_layout=True,
    )
    im = ax.imshow(
        diff,
        aspect="auto",
        origin="lower",
        extent=[
            epochs.times[0], epochs.times[-1],
            -0.5, len(epochs.ch_names) - 0.5,
        ],
        interpolation=INTERPOLATION,
        resample=False,
        cmap="RdBu_r",
        vmin=-v,
        vmax=v,
    )
    add_timing_annotations(ax, shade_current_window=True)
    ax.set_yticks(np.arange(len(epochs.ch_names)))
    labels = [
        ch + (" *" if ch in PREVIOUS_BEST_AVAILABLE else "")
        for ch in epochs.ch_names
    ]
    ax.set_yticklabels(labels)
    ax.set_xlabel("Time from LSL marker, s")
    ax.set_ylabel("Channel (* = previous best)")
    ax.set_title(
        f"{PATIENT_ID} | ERP difference: open_hand − all_other\n"
        + timing_caption()
    )
    ax.legend(handles=timing_handles(True), loc="upper right", framealpha=0.9)
    fig.colorbar(im, ax=ax, label="Amplitude difference")
    save_figure(fig, out_path)


def deterministic_subsample(data, max_rows, seed):
    if len(data) <= max_rows:
        return data
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(data), size=max_rows, replace=False))
    return data[idx]


def plot_per_channel_trial_cards(epochs: mne.Epochs, out_dir: Path):
    open_data, other_data, _ = binary_data(epochs)
    times = epochs.times

    for ch_idx, ch_name in enumerate(epochs.ch_names):
        open_ch = deterministic_subsample(
            open_data[:, ch_idx, :],
            MAX_TRIALS_PER_CLASS_ON_FIGURE,
            RANDOM_SEED + ch_idx,
        )
        other_ch = deterministic_subsample(
            other_data[:, ch_idx, :],
            MAX_TRIALS_PER_CLASS_ON_FIGURE,
            RANDOM_SEED + 1000 + ch_idx,
        )

        v = robust_symmetric_limit([open_ch, other_ch], q=0.99)

        fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)

        for ax, arr, title in [
            (axes[0, 0], open_ch, f"open_hand trials (shown {len(open_ch)})"),
            (axes[0, 1], other_ch, f"all_other trials (shown {len(other_ch)})"),
        ]:
            im = ax.imshow(
                arr,
                aspect="auto",
                origin="lower",
                extent=[times[0], times[-1], 0, len(arr)],
                interpolation=INTERPOLATION,
                cmap="RdBu_r",
                vmin=-v,
                vmax=v,
            )
            add_timing_annotations(ax, shade_current_window=True)
            ax.set_title(title)
            ax.set_xlabel("Time from LSL marker, s")
            ax.set_ylabel("Displayed trials")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

        axes[1, 0].plot(times, open_data[:, ch_idx, :].mean(axis=0), label="open_hand")
        axes[1, 0].plot(times, other_data[:, ch_idx, :].mean(axis=0), label="all_other")
        add_timing_annotations(axes[1, 0], shade_current_window=True)
        axes[1, 0].set_title("Class means")
        axes[1, 0].set_xlabel("Time from LSL marker, s")
        axes[1, 0].set_ylabel("Amplitude")
        axes[1, 0].legend()

        diff = (
            open_data[:, ch_idx, :].mean(axis=0)
            - other_data[:, ch_idx, :].mean(axis=0)
        )
        axes[1, 1].plot(times, diff)
        add_timing_annotations(axes[1, 1], shade_current_window=True)
        axes[1, 1].set_title("open_hand − all_other")
        axes[1, 1].set_xlabel("Time from LSL marker, s")
        axes[1, 1].set_ylabel("Amplitude difference")

        fig.suptitle(
            f"{PATIENT_ID} | {ch_name} | ERP/trial screening"
            + (" | previous best" if ch_name in PREVIOUS_BEST_AVAILABLE else "")
            + "\n" + timing_caption()
        )
        save_figure(fig, out_dir / f"channel_{safe_name(ch_name)}__erp_trials")


if RUN_ERP:
    plot_erp_overlay_pages(patient_epochs_bc, DIRS["erp"])
    plot_erp_difference_heatmap(
        patient_epochs_bc,
        DIRS["erp"] / "02_all_channels_erp_difference_heatmap",
    )
    plot_per_channel_trial_cards(patient_epochs_bc, DIRS["erp_cards"])
    print("Saved ERP screening to:", DIRS["erp"])


Saved ERP screening to: /trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/stage3_input_screening/s11/02_erp_channel_screening


## 5. Этап C — средние TFR по всем каналам

In [10]:

def compute_average_tfr(epochs: mne.Epochs, positive: bool):
    labels = (epochs.events[:, 2] == 9)
    selected = epochs[labels if positive else ~labels]
    freqs = np.linspace(0.1, 120.0, TFR_FREQS)
    n_cycles = freqs / 2.0

    return mne.time_frequency.tfr_morlet(
        selected,
        freqs=freqs,
        n_cycles=n_cycles,
        use_fft=True,
        return_itc=False,
        average=True,
        decim=TFR_DECIM,
        n_jobs=N_JOBS,
        verbose="ERROR",
    )


def plot_tfr_pages(tfr_open, tfr_other, out_dir: Path):
    open_db = power_to_db(tfr_open.data)
    other_db = power_to_db(tfr_other.data)
    diff_db = open_db - other_db
    times = tfr_open.times
    freqs = tfr_open.freqs

    n_pages = math.ceil(len(tfr_open.ch_names) / CHANNELS_PER_PAGE)

    for page in range(n_pages):
        start = page * CHANNELS_PER_PAGE
        stop = min((page + 1) * CHANNELS_PER_PAGE, len(tfr_open.ch_names))
        idxs = list(range(start, stop))

        fig, axes = plt.subplots(
            len(idxs), 3,
            figsize=(16, max(3.5, 2.8 * len(idxs))),
            constrained_layout=True,
        )
        if len(idxs) == 1:
            axes = np.expand_dims(axes, axis=0)

        page_abs = np.concatenate([
            open_db[idxs].ravel(),
            other_db[idxs].ravel(),
        ])
        abs_vmin, abs_vmax = np.quantile(page_abs, [0.01, 0.99])
        diff_v = robust_symmetric_limit([diff_db[idxs]], q=0.99)

        for row, ch_idx in enumerate(idxs):
            panels = [
                (open_db[ch_idx], "open_hand log-power", "viridis", abs_vmin, abs_vmax),
                (other_db[ch_idx], "all_other log-power", "viridis", abs_vmin, abs_vmax),
                (diff_db[ch_idx], "open − other, dB", "RdBu_r", -diff_v, diff_v),
            ]
            for col, (arr, title, cmap, vmin, vmax) in enumerate(panels):
                ax = axes[row, col]
                im = ax.imshow(
                    arr,
                    aspect="auto",
                    origin="lower",
                    extent=[times[0], times[-1], freqs[0], freqs[-1]],
                    interpolation=INTERPOLATION,
                    cmap=cmap,
                    vmin=vmin,
                    vmax=vmax,
                )
                add_timing_annotations(ax)
                add_decoder_rectangle(ax)
                ch_name = tfr_open.ch_names[ch_idx]
                ax.set_title(
                    f"{ch_name} | {title}"
                    + (" | previous best" if ch_name in PREVIOUS_BEST_AVAILABLE else "")
                )
                ax.set_xlabel("Time from LSL marker, s")
                ax.set_ylabel("Frequency, Hz")
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

        axes[0, 0].legend(
            handles=timing_handles(True),
            loc="upper right",
            framealpha=0.9,
        )
        fig.suptitle(
            f"{PATIENT_ID} | binary average TFR | page {page+1}/{n_pages}\n"
            + timing_caption()
        )
        save_figure(fig, out_dir / f"01_binary_tfr_page_{page+1:02d}")


def plot_per_channel_tfr_cards(tfr_open, tfr_other, out_dir: Path):
    open_db = power_to_db(tfr_open.data)
    other_db = power_to_db(tfr_other.data)
    diff_db = open_db - other_db
    times = tfr_open.times
    freqs = tfr_open.freqs

    for ch_idx, ch_name in enumerate(tfr_open.ch_names):
        abs_values = np.concatenate([
            open_db[ch_idx].ravel(), other_db[ch_idx].ravel()
        ])
        abs_vmin, abs_vmax = np.quantile(abs_values, [0.01, 0.99])
        diff_v = robust_symmetric_limit([diff_db[ch_idx]], q=0.99)

        fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
        panels = [
            (open_db[ch_idx], "open_hand", "viridis", abs_vmin, abs_vmax),
            (other_db[ch_idx], "all_other", "viridis", abs_vmin, abs_vmax),
            (diff_db[ch_idx], "open − other, dB", "RdBu_r", -diff_v, diff_v),
        ]
        for ax, (arr, title, cmap, vmin, vmax) in zip(axes, panels):
            im = ax.imshow(
                arr,
                aspect="auto",
                origin="lower",
                extent=[times[0], times[-1], freqs[0], freqs[-1]],
                interpolation=INTERPOLATION,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
            )
            add_timing_annotations(ax)
            add_decoder_rectangle(ax)
            ax.set_title(title)
            ax.set_xlabel("Time from LSL marker, s")
            ax.set_ylabel("Frequency, Hz")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

        axes[0].legend(handles=timing_handles(True), loc="upper right", framealpha=0.9)
        fig.suptitle(
            f"{PATIENT_ID} | {ch_name} | TFR screening"
            + (" | previous best" if ch_name in PREVIOUS_BEST_AVAILABLE else "")
            + "\n" + timing_caption()
        )
        save_figure(fig, out_dir / f"channel_{safe_name(ch_name)}__tfr")


if RUN_TFR:
    tfr_open = compute_average_tfr(patient_epochs_bc, positive=True)
    tfr_other = compute_average_tfr(patient_epochs_bc, positive=False)
    plot_tfr_pages(tfr_open, tfr_other, DIRS["tfr"])
    plot_per_channel_tfr_cards(tfr_open, tfr_other, DIRS["tfr_cards"])
    print("Saved TFR screening to:", DIRS["tfr"])


Saved TFR screening to: /trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/stage3_input_screening/s11/03_tfr_channel_screening


## 6. Этап D — количественные discriminability maps

In [11]:

def cohen_d_map(x_pos, x_neg):
    mean_pos = np.mean(x_pos, axis=0)
    mean_neg = np.mean(x_neg, axis=0)
    var_pos = np.var(x_pos, axis=0, ddof=1)
    var_neg = np.var(x_neg, axis=0, ddof=1)
    n_pos = len(x_pos)
    n_neg = len(x_neg)

    pooled = np.sqrt(
        ((n_pos - 1) * var_pos + (n_neg - 1) * var_neg)
        / max(n_pos + n_neg - 2, 1)
    )
    return np.divide(
        mean_pos - mean_neg,
        pooled,
        out=np.zeros_like(mean_pos),
        where=pooled > 0,
    )


def auc_map_chunked(features_2d, labels, chunk_size):
    labels = np.asarray(labels).astype(int)
    n_pos = int(labels.sum())
    n_neg = int(len(labels) - n_pos)
    result = np.empty(features_2d.shape[1], dtype=float)

    for start in range(0, features_2d.shape[1], chunk_size):
        stop = min(start + chunk_size, features_2d.shape[1])
        block = features_2d[:, start:stop]
        ranks = rankdata(block, axis=0)
        rank_sum_pos = ranks[labels == 1].sum(axis=0)
        result[start:stop] = (
            rank_sum_pos - n_pos * (n_pos + 1) / 2.0
        ) / (n_pos * n_neg)

    return result


def compute_channel_discriminability(epochs: mne.Epochs, ch_idx: int):
    ch_name = epochs.ch_names[ch_idx]
    cache_path = DIRS["cache"] / f"disc_{safe_name(ch_name)}.npz"

    if cache_path.exists():
        cached = np.load(cache_path)
        return {
            "times": cached["times"],
            "freqs": cached["freqs"],
            "cohen_d": cached["cohen_d"],
            "auc": cached["auc"] if "auc" in cached.files else None,
        }

    data = epochs.get_data()[:, ch_idx:ch_idx+1, :]
    labels = (epochs.events[:, 2] == 9).astype(int)
    freqs = np.linspace(0.1, 120.0, TFR_FREQS)
    n_cycles = freqs / 2.0

    power = mne.time_frequency.tfr_array_morlet(
        data,
        sfreq=epochs.info["sfreq"],
        freqs=freqs,
        n_cycles=n_cycles,
        output="power",
        decim=TFR_DECIM,
        n_jobs=N_JOBS,
        verbose="ERROR",
    )[:, 0]

    log_power = power_to_db(power)
    d_map = cohen_d_map(log_power[labels == 1], log_power[labels == 0])

    auc_map = None
    if COMPUTE_AUC:
        auc_flat = auc_map_chunked(
            log_power.reshape(log_power.shape[0], -1),
            labels,
            AUC_CHUNK_SIZE,
        )
        auc_map = auc_flat.reshape(log_power.shape[1:])

    times = epochs.times[::TFR_DECIM]
    save_kwargs = {
        "times": times,
        "freqs": freqs,
        "cohen_d": d_map,
    }
    if auc_map is not None:
        save_kwargs["auc"] = auc_map
    np.savez_compressed(cache_path, **save_kwargs)

    return {
        "times": times,
        "freqs": freqs,
        "cohen_d": d_map,
        "auc": auc_map,
    }


def plot_discriminability_card(ch_name, result, out_dir):
    times = result["times"]
    freqs = result["freqs"]
    d_map = result["cohen_d"]

    panels = [(d_map, "Cohen's d", "RdBu_r", True)]
    if result["auc"] is not None:
        panels.append((np.abs(result["auc"] - 0.5), "|AUC − 0.5|", "magma", False))

    fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 5), constrained_layout=True)
    axes = np.atleast_1d(axes)

    for ax, (arr, title, cmap, symmetric) in zip(axes, panels):
        if symmetric:
            v = robust_symmetric_limit([arr], q=0.99)
            vmin, vmax = -v, v
        else:
            vmin, vmax = 0.0, max(0.03, float(np.quantile(arr, 0.99)))

        im = ax.imshow(
            arr,
            aspect="auto",
            origin="lower",
            extent=[times[0], times[-1], freqs[0], freqs[-1]],
            interpolation=INTERPOLATION,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        add_timing_annotations(ax)
        add_decoder_rectangle(ax)
        ax.set_title(title)
        ax.set_xlabel("Time from LSL marker, s")
        ax.set_ylabel("Frequency, Hz")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    axes[0].legend(handles=timing_handles(True), loc="upper right", framealpha=0.9)
    fig.suptitle(
        f"{PATIENT_ID} | {ch_name} | discriminability"
        + (" | previous best" if ch_name in PREVIOUS_BEST_AVAILABLE else "")
        + "\n" + timing_caption()
    )
    save_figure(fig, out_dir / f"channel_{safe_name(ch_name)}__discriminability")


def mask_for_region(times, freqs, time_window, freq_range):
    time_mask = (times >= time_window[0]) & (times <= time_window[1])
    freq_mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])
    return np.outer(freq_mask, time_mask)


if RUN_DISCRIMINABILITY:
    discriminability = {}
    for ch_idx, ch_name in enumerate(patient_epochs_bc.ch_names):
        print(f"Discriminability {ch_idx+1}/{len(patient_epochs_bc.ch_names)}: {ch_name}")
        result = compute_channel_discriminability(patient_epochs_bc, ch_idx)
        discriminability[ch_name] = result
        plot_discriminability_card(ch_name, result, DIRS["disc_cards"])

    first = next(iter(discriminability.values()))
    d_stack = np.stack([np.abs(x["cohen_d"]) for x in discriminability.values()])
    median_d = np.median(d_stack, axis=0)

    aggregate_panels = [(median_d, "Median |Cohen's d| across channels")]
    if COMPUTE_AUC:
        auc_stack = np.stack([
            np.abs(x["auc"] - 0.5) for x in discriminability.values()
        ])
        median_auc = np.median(auc_stack, axis=0)
        aggregate_panels.append((median_auc, "Median |AUC − 0.5| across channels"))

    fig, axes = plt.subplots(
        1, len(aggregate_panels),
        figsize=(7 * len(aggregate_panels), 5),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    for ax, (arr, title) in zip(axes, aggregate_panels):
        im = ax.imshow(
            arr,
            aspect="auto",
            origin="lower",
            extent=[
                first["times"][0], first["times"][-1],
                first["freqs"][0], first["freqs"][-1],
            ],
            interpolation=INTERPOLATION,
            cmap="magma",
            vmin=0,
            vmax=max(0.03, float(np.quantile(arr, 0.99))),
        )
        add_timing_annotations(ax)
        add_decoder_rectangle(ax)
        ax.set_title(title)
        ax.set_xlabel("Time from LSL marker, s")
        ax.set_ylabel("Frequency, Hz")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    axes[0].legend(handles=timing_handles(True), loc="upper right", framealpha=0.9)
    fig.suptitle(f"{PATIENT_ID} | aggregate discriminability\n" + timing_caption())
    save_figure(fig, DIRS["disc"] / "01_aggregate_discriminability")


Discriminability 1/11: Fp1
Discriminability 2/11: F7
Discriminability 3/11: Ft7
Discriminability 4/11: T3
Discriminability 5/11: F8
Discriminability 6/11: Fp2
Discriminability 7/11: Fpz
Discriminability 8/11: F3
Discriminability 9/11: Fc3
Discriminability 10/11: Tp7
Discriminability 11/11: Fc4


## 7. Этап E — ranking и стабильность между сессиями

In [12]:

def canonical_channel_metrics(discriminability, qc_df):
    rows = []
    for ch_name, result in discriminability.items():
        region = mask_for_region(
            result["times"],
            result["freqs"],
            CURRENT_TIME_WINDOW,
            CURRENT_FREQ_RANGE,
        )
        abs_d = np.abs(result["cohen_d"])
        row = {
            "channel": ch_name,
            "previous_best": ch_name in PREVIOUS_BEST_AVAILABLE,
            "mean_abs_cohen_d_current_window": float(np.mean(abs_d[region])),
            "p95_abs_cohen_d_current_window": float(np.quantile(abs_d[region], 0.95)),
        }
        if result["auc"] is not None:
            auc_strength = np.abs(result["auc"] - 0.5)
            row.update({
                "mean_abs_auc_minus_0p5_current_window": float(np.mean(auc_strength[region])),
                "p95_abs_auc_minus_0p5_current_window": float(np.quantile(auc_strength[region], 0.95)),
            })
        rows.append(row)

    ranking = pd.DataFrame(rows).merge(qc_df, on=["channel", "previous_best"], how="left")

    ranking["rank_mean_abs_d"] = ranking[
        "mean_abs_cohen_d_current_window"
    ].rank(ascending=False, method="average")

    rank_columns = ["rank_mean_abs_d"]
    if "mean_abs_auc_minus_0p5_current_window" in ranking:
        ranking["rank_mean_auc"] = ranking[
            "mean_abs_auc_minus_0p5_current_window"
        ].rank(ascending=False, method="average")
        rank_columns.append("rank_mean_auc")

    ranking["rank_low_artifact_fraction"] = ranking[
        "fraction_epochs_ptp_above_config_threshold"
    ].rank(ascending=True, method="average")
    rank_columns.append("rank_low_artifact_fraction")

    ranking["consensus_screening_rank"] = ranking[rank_columns].mean(axis=1)
    ranking = ranking.sort_values("consensus_screening_rank").reset_index(drop=True)
    return ranking


def compute_session_erp_effect(session_obj):
    epochs = session_obj["epochs"].copy().apply_baseline(PIPELINE_BASELINE_MARKER)
    data = epochs.get_data()
    labels = (epochs.events[:, 2] == 9)
    times = epochs.times
    window = (times >= CURRENT_TIME_WINDOW[0]) & (times <= CURRENT_TIME_WINDOW[1])
    baseline = (
        (times >= PIPELINE_BASELINE_MARKER[0])
        & (times <= PIPELINE_BASELINE_MARKER[1])
    )

    rows = []
    for ch_idx, ch_name in enumerate(epochs.ch_names):
        diff = (
            data[labels, ch_idx, :].mean(axis=0)
            - data[~labels, ch_idx, :].mean(axis=0)
        )
        baseline_scale = np.std(data[:, ch_idx, :][:, baseline])
        effect = np.mean(np.abs(diff[window])) / max(baseline_scale, np.finfo(float).eps)
        rows.append({
            "session": session_obj["session"],
            "channel": ch_name,
            "erp_effect_normalized": float(effect),
        })
    return rows


if RUN_DISCRIMINABILITY:
    if "qc_df" not in globals():
        qc_df = compute_qc_table(patient_epochs)

    ranking_df = canonical_channel_metrics(discriminability, qc_df)

    if RUN_SESSION_STABILITY:
        stability_rows = []
        for obj in session_objects:
            stability_rows.extend(compute_session_erp_effect(obj))
        stability_df = pd.DataFrame(stability_rows)
        stability_pivot = stability_df.pivot(
            index="channel",
            columns="session",
            values="erp_effect_normalized",
        ).reindex(AVAILABLE_CHANNELS)
        stability_pivot.to_csv(DIRS["ranking"] / "session_erp_effect_by_channel.csv")

        ranking_df = ranking_df.merge(
            stability_df.groupby("channel")["erp_effect_normalized"]
            .agg(["median", "min", "max", "std"])
            .reset_index()
            .rename(columns={
                "median": "session_median_erp_effect",
                "min": "session_min_erp_effect",
                "max": "session_max_erp_effect",
                "std": "session_std_erp_effect",
            }),
            on="channel",
            how="left",
        )

        fig, ax = plt.subplots(
            figsize=(max(9, 2.2 * len(stability_pivot.columns)), max(5, 0.45 * len(stability_pivot))),
            constrained_layout=True,
        )
        im = ax.imshow(
            stability_pivot.to_numpy(),
            aspect="auto",
            origin="lower",
            interpolation=INTERPOLATION,
            cmap="viridis",
        )
        ax.set_xticks(np.arange(len(stability_pivot.columns)))
        ax.set_xticklabels(stability_pivot.columns, rotation=45, ha="right")
        ax.set_yticks(np.arange(len(stability_pivot.index)))
        ax.set_yticklabels([
            ch + (" *" if ch in PREVIOUS_BEST_AVAILABLE else "")
            for ch in stability_pivot.index
        ])
        ax.set_title(
            f"{PATIENT_ID} | normalized ERP class difference by session\n"
            "* = previous best; high only in one session is a warning"
        )
        ax.set_xlabel("Session")
        ax.set_ylabel("Channel")
        fig.colorbar(im, ax=ax, label="Normalized ERP difference")
        save_figure(fig, DIRS["ranking"] / "01_session_stability_heatmap")

    ranking_df.to_csv(DIRS["ranking"] / "channel_screening_ranking.csv", index=False)
    display(ranking_df)

    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    ordered = ranking_df.sort_values("consensus_screening_rank", ascending=False)
    ax.barh(ordered["channel"], ordered["consensus_screening_rank"])
    ax.set_xlabel("Consensus screening rank (lower is better)")
    ax.set_title(
        f"{PATIENT_ID} | channel screening rank\n"
        "Signal effect + AUC + artifact audit; not classifier accuracy"
    )
    save_figure(fig, DIRS["ranking"] / "02_channel_consensus_rank")


Applying baseline correction (mode: mean)
Applying baseline correction (mode: mean)


,channel,previous_best,mean_abs_cohen_d_current_window,p95_abs_cohen_d_current_window,mean_abs_auc_minus_0p5_current_window,p95_abs_auc_minus_0p5_current_window,median_peak_to_peak_v,p95_peak_to_peak_v,max_peak_to_peak_v,median_baseline_rms_v,fraction_epochs_ptp_above_config_threshold,rank_mean_abs_d,rank_mean_auc,rank_low_artifact_fraction,consensus_screening_rank,session_median_erp_effect,session_min_erp_effect,session_max_erp_effect,session_std_erp_effect
0,F8,True,0.114019,0.316103,0.034526,0.093511,0.000167,0.000210,0.000593,0.000026,0.000000,1.0,1.0,2.0,1.333333,0.200710,0.191549,0.209870,0.012955
1,Fp1,True,0.109515,0.348840,0.033045,0.106225,0.000211,0.000280,0.000590,0.000028,0.000000,2.0,2.0,2.0,2.000000,1.157150,1.057024,1.257276,0.141600
2,F3,True,0.104157,0.285849,0.032092,0.087221,0.000075,0.000106,0.000557,0.000012,0.000000,3.0,3.0,2.0,2.666667,0.297153,0.284256,0.310049,0.018239
3,Ft7,False,0.103249,0.279210,0.030802,0.084578,0.000117,0.000166,0.000756,0.000018,0.001869,4.0,6.0,7.0,5.666667,0.525661,0.418954,0.632367,0.150905
4,Fp2,True,0.102389,0.254038,0.030984,0.076583,0.000081,0.000115,0.000768,0.000012,0.001869,6.0,4.0,7.0,5.666667,0.570998,0.565884,0.576111,0.007232
5,T3,False,0.102607,0.246849,0.030909,0.074138,0.000117,0.000155,0.000710,0.000017,0.001869,5.0,5.0,7.0,5.666667,0.538740,0.486298,0.591183,0.074165
6,F7,True,0.092166,0.247714,0.027321,0.074758,0.000286,0.000394,0.000819,0.000042,0.001869,8.0,8.0,7.0,7.666667,0.628531,0.601622,0.655439,0.038054
7,Fc3,False,0.086402,0.207301,0.025550,0.062036,0.000094,0.000127,0.001001,0.000014,0.001869,9.0,9.0,7.0,8.333333,0.539177,0.531281,0.547074,0.011168
8,Fc4,False,0.094937,0.249217,0.028757,0.074085,0.000100,0.000200,0.001705,0.000016,0.013084,7.0,7.0,11.0,8.333333,1.059060,0.816749,1.301371,0.342680
9,Tp7,True,0.081940,0.216469,0.024286,0.065507,0.000329,0.000478,0.001289,0.000053,0.001869,10.0,10.0,7.0,9.000000,0.733409,0.632443,0.834376,0.142788


## 8. Этап F — screening временных окон

In [13]:

def region_metric_rows(discriminability, time_windows, frequency_bands):
    rows = []
    for ch_name, result in discriminability.items():
        for time_id, time_window in time_windows.items():
            for freq_id, freq_range in frequency_bands.items():
                region = mask_for_region(
                    result["times"], result["freqs"], time_window, freq_range
                )
                row = {
                    "channel": ch_name,
                    "previous_best": ch_name in PREVIOUS_BEST_AVAILABLE,
                    "time_id": time_id,
                    "time_start_marker_s": time_window[0],
                    "time_end_marker_s": time_window[1],
                    "freq_id": freq_id,
                    "freq_low_hz": freq_range[0],
                    "freq_high_hz": freq_range[1],
                    "mean_abs_cohen_d": float(np.mean(np.abs(result["cohen_d"])[region])),
                }
                if result["auc"] is not None:
                    row["mean_abs_auc_minus_0p5"] = float(
                        np.mean(np.abs(result["auc"] - 0.5)[region])
                    )
                rows.append(row)
    return pd.DataFrame(rows)


if RUN_TIME_SCREENING and RUN_DISCRIMINABILITY:
    all_region_scores = region_metric_rows(
        discriminability,
        TIME_WINDOWS,
        FREQUENCY_BANDS,
    )
    all_region_scores.to_csv(
        DIRS["time"] / "all_channel_time_frequency_region_scores.csv",
        index=False,
    )

    time_scores = all_region_scores[
        all_region_scores["freq_id"] == "F0_current_0p1_59p4"
    ].copy()
    time_scores.to_csv(
        DIRS["time"] / "time_window_scores_at_current_frequency_range.csv",
        index=False,
    )

    pivot = time_scores.pivot(
        index="channel",
        columns="time_id",
        values="mean_abs_cohen_d",
    ).reindex(AVAILABLE_CHANNELS)

    fig, ax = plt.subplots(
        figsize=(12, max(5, 0.45 * len(pivot))),
        constrained_layout=True,
    )
    im = ax.imshow(
        pivot.to_numpy(),
        aspect="auto",
        origin="lower",
        interpolation=INTERPOLATION,
        cmap="viridis",
    )
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([
        ch + (" *" if ch in PREVIOUS_BEST_AVAILABLE else "")
        for ch in pivot.index
    ])
    ax.set_title(
        f"{PATIENT_ID} | time-window screening at 0.1–59.4 Hz\n"
        "Mean |Cohen's d|; * = previous best; not classifier accuracy"
    )
    ax.set_xlabel("Predefined time window")
    ax.set_ylabel("Channel")
    fig.colorbar(im, ax=ax, label="Mean |Cohen's d|")
    save_figure(fig, DIRS["time"] / "01_time_window_by_channel_heatmap")

    time_summary = (
        time_scores.groupby("time_id")
        .agg(
            median_across_channels=("mean_abs_cohen_d", "median"),
            q25=("mean_abs_cohen_d", lambda x: x.quantile(0.25)),
            q75=("mean_abs_cohen_d", lambda x: x.quantile(0.75)),
        )
        .reset_index()
        .sort_values("median_across_channels", ascending=False)
    )
    time_summary.to_csv(DIRS["time"] / "time_window_summary.csv", index=False)
    display(time_summary)

    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    ax.bar(time_summary["time_id"], time_summary["median_across_channels"])
    ax.set_ylabel("Median channel mean |Cohen's d|")
    ax.set_title(
        f"{PATIENT_ID} | predefined time windows\n"
        "Screening only; final choice requires fixed-fold ablation"
    )
    ax.tick_params(axis="x", rotation=35)
    save_figure(fig, DIRS["time"] / "02_time_window_summary")


,time_id,median_across_channels,q25,q75
0,T0_current_marker_0_1,0.102389,0.089284,0.103703
2,T2_robot_0_0p5,0.101347,0.092215,0.105597
1,T1_robot_0_1,0.097235,0.088658,0.103075
3,T3_robot_0_1p5,0.091961,0.083651,0.093425
4,T4_robot_0_2,0.087120,0.084815,0.093092


## 9. Этап G — screening частотных диапазонов

In [14]:

if RUN_FREQUENCY_SCREENING and RUN_DISCRIMINABILITY:
    if "all_region_scores" not in globals():
        all_region_scores = region_metric_rows(
            discriminability,
            TIME_WINDOWS,
            FREQUENCY_BANDS,
        )

    frequency_scores = all_region_scores[
        all_region_scores["time_id"] == "T0_current_marker_0_1"
    ].copy()
    frequency_scores.to_csv(
        DIRS["freq"] / "frequency_band_scores_at_current_time_window.csv",
        index=False,
    )

    pivot = frequency_scores.pivot(
        index="channel",
        columns="freq_id",
        values="mean_abs_cohen_d",
    ).reindex(AVAILABLE_CHANNELS)

    fig, ax = plt.subplots(
        figsize=(13, max(5, 0.45 * len(pivot))),
        constrained_layout=True,
    )
    im = ax.imshow(
        pivot.to_numpy(),
        aspect="auto",
        origin="lower",
        interpolation=INTERPOLATION,
        cmap="viridis",
    )
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([
        ch + (" *" if ch in PREVIOUS_BEST_AVAILABLE else "")
        for ch in pivot.index
    ])
    ax.set_title(
        f"{PATIENT_ID} | frequency-band screening at marker 0–1 s\n"
        "Mean |Cohen's d|; * = previous best; not classifier accuracy"
    )
    ax.set_xlabel("Predefined frequency band")
    ax.set_ylabel("Channel")
    fig.colorbar(im, ax=ax, label="Mean |Cohen's d|")
    save_figure(fig, DIRS["freq"] / "01_frequency_band_by_channel_heatmap")

    freq_summary = (
        frequency_scores.groupby("freq_id")
        .agg(
            median_across_channels=("mean_abs_cohen_d", "median"),
            q25=("mean_abs_cohen_d", lambda x: x.quantile(0.25)),
            q75=("mean_abs_cohen_d", lambda x: x.quantile(0.75)),
        )
        .reset_index()
        .sort_values("median_across_channels", ascending=False)
    )
    freq_summary.to_csv(DIRS["freq"] / "frequency_band_summary.csv", index=False)
    display(freq_summary)

    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
    ax.bar(freq_summary["freq_id"], freq_summary["median_across_channels"])
    ax.set_ylabel("Median channel mean |Cohen's d|")
    ax.set_title(
        f"{PATIENT_ID} | predefined frequency bands\n"
        "Screening only; final choice requires fixed-fold ablation"
    )
    ax.tick_params(axis="x", rotation=35)
    save_figure(fig, DIRS["freq"] / "02_frequency_band_summary")


,freq_id,median_across_channels,q25,q75
1,F1_low_0p1_30,0.116242,0.103241,0.133199
2,F2_alpha_beta_8_30,0.107311,0.088633,0.118826
0,F0_current_0p1_59p4,0.102389,0.089284,0.103703
4,F4_full_0p1_120,0.083095,0.078889,0.084405
3,F3_low_gamma_30_59p4,0.079641,0.070335,0.082225
5,F5_upper_59p4_120,0.066869,0.064117,0.070903


## 10. Этап H — selected против control-кандидатов

In [15]:

def choose_equal_size_screening_sets(ranking_df):
    previous = [ch for ch in PREVIOUS_BEST_AVAILABLE if ch in AVAILABLE_CHANNELS]
    remaining = [ch for ch in AVAILABLE_CHANNELS if ch not in previous]

    if MANUAL_SELECTED_CHANNELS is not None:
        selected = [ch for ch in MANUAL_SELECTED_CHANNELS if ch in AVAILABLE_CHANNELS]
    else:
        selected = previous

    if MANUAL_CONTROL_CHANNELS is not None:
        control = [ch for ch in MANUAL_CONTROL_CHANNELS if ch in AVAILABLE_CHANNELS]
    else:
        control = remaining

    if not selected or not control:
        return selected, control, [], []

    k = min(len(selected), len(control))
    rank_lookup = ranking_df.set_index("channel")["consensus_screening_rank"]

    # Для честного визуального сравнения групп одинакового размера:
    # лучшие по screening rank внутри previous selected
    # и худшие по screening rank среди оставшихся каналов.
    selected_equal = sorted(selected, key=lambda ch: rank_lookup.get(ch, np.inf))[:k]
    control_equal = sorted(control, key=lambda ch: rank_lookup.get(ch, -np.inf), reverse=True)[:k]
    return selected, control, selected_equal, control_equal


if RUN_SELECTED_CONTROL_REVIEW and RUN_DISCRIMINABILITY:
    selected_all, control_pool, selected_equal, control_equal = (
        choose_equal_size_screening_sets(ranking_df)
    )

    role_rows = []
    for _, row in ranking_df.iterrows():
        ch = row["channel"]
        if ch in selected_equal:
            suggestion = "selected_equal_size"
        elif ch in control_equal:
            suggestion = "control_equal_size"
        elif ch in selected_all:
            suggestion = "previous_selected_not_in_equal_subset"
        elif ch in control_pool:
            suggestion = "remaining_candidate"
        else:
            suggestion = "unassigned"

        role_rows.append({
            "channel": ch,
            "previous_best": ch in PREVIOUS_BEST_AVAILABLE,
            "consensus_screening_rank": row["consensus_screening_rank"],
            "suggested_role": suggestion,
            "manual_role": "",
            "manual_comment": "",
        })

    role_df = pd.DataFrame(role_rows)
    role_df.to_csv(
        DIRS["control"] / "selected_control_review_template.csv",
        index=False,
    )
    display(role_df)

    plot_df = ranking_df.copy().sort_values(
        "consensus_screening_rank",
        ascending=False,
    )
    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    ax.barh(plot_df["channel"], plot_df["consensus_screening_rank"])

    for y, (_, row) in enumerate(plot_df.iterrows()):
        ch = row["channel"]
        if ch in selected_equal:
            label = "selected"
        elif ch in control_equal:
            label = "control"
        elif ch in selected_all:
            label = "previous selected"
        else:
            label = "candidate"
        ax.text(row["consensus_screening_rank"], y, "  " + label, va="center", fontsize=8)

    ax.set_xlabel("Consensus screening rank (lower is better)")
    ax.set_title(
        f"{PATIENT_ID} | equal-size selected/control proposal\n"
        f"selected={selected_equal}; control={control_equal}"
    )
    save_figure(fig, DIRS["control"] / "01_selected_control_proposal")

    note = (
        f"Full previous selected: {selected_all}\n"
        f"Remaining control pool: {control_pool}\n"
        f"Equal-size selected subset: {selected_equal}\n"
        f"Equal-size control subset: {control_equal}\n\n"
        "This is a screening proposal only. Edit manual_role in the CSV after "
        "reviewing ERP, trial heatmaps, TFR and session stability."
    )
    (DIRS["control"] / "HOW_EQUAL_SIZE_SETS_WERE_FORMED.txt").write_text(
        note,
        encoding="utf-8",
    )
    print(note)


,channel,previous_best,consensus_screening_rank,suggested_role,manual_role,manual_comment
0,F8,True,1.333333,selected_equal_size,,
1,Fp1,True,2.000000,selected_equal_size,,
2,F3,True,2.666667,selected_equal_size,,
3,Ft7,False,5.666667,control_equal_size,,
4,Fp2,True,5.666667,selected_equal_size,,
5,T3,False,5.666667,control_equal_size,,
6,F7,True,7.666667,previous_selected_not_in_equal_subset,,
7,Fc3,False,8.333333,control_equal_size,,
8,Fc4,False,8.333333,control_equal_size,,
9,Tp7,True,9.000000,previous_selected_not_in_equal_subset,,


Full previous selected: ['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']
Remaining control pool: ['Ft7', 'T3', 'Fc3', 'Fc4']
Equal-size selected subset: ['F8', 'Fp1', 'F3', 'Fp2']
Equal-size control subset: ['Fc3', 'Fc4', 'Ft7', 'T3']

This is a screening proposal only. Edit manual_role in the CSV after reviewing ERP, trial heatmaps, TFR and session stability.


## 11. Этап I — финальный лист ручного выбора

In [16]:

if RUN_DISCRIMINABILITY:
    final_sheet = ranking_df.copy()
    final_sheet.insert(1, "manual_keep_for_ablation", "")
    final_sheet.insert(2, "manual_use_as_control", "")
    final_sheet.insert(3, "manual_exclude_reason", "")
    final_sheet.insert(4, "manual_comment", "")
    final_sheet.to_csv(
        DIRS["final"] / "channel_selection_sheet_EDIT_ME.csv",
        index=False,
    )

    recommended_time = None
    recommended_freq = None
    if "time_summary" in globals() and len(time_summary):
        recommended_time = time_summary.iloc[0]["time_id"]
    if "freq_summary" in globals() and len(freq_summary):
        recommended_freq = freq_summary.iloc[0]["freq_id"]

    summary_lines = [
        f"Patient: {PATIENT_ID}",
        "",
        "This notebook completed signal-level screening only.",
        "",
        f"Previous best channels from config: {PREVIOUS_BEST_AVAILABLE}",
        f"Top screening channels: {ranking_df.head(min(5, len(ranking_df)))['channel'].tolist()}",
        f"Highest median predefined time window: {recommended_time}",
        f"Highest median predefined frequency band: {recommended_freq}",
        "",
        "Before moving to model ablation:",
        "1. Open 01_inventory_and_qc and exclude clearly unstable/artifact channels.",
        "2. Inspect every per-channel ERP/trial card.",
        "3. Inspect every per-channel TFR and discriminability card.",
        "4. Check session stability; do not select a channel driven by one session only.",
        "5. Edit channel_selection_sheet_EDIT_ME.csv.",
        "6. Freeze selected/control/time/frequency candidates.",
        "7. Run a separate fixed-fold SVM ablation; do not report these screening scores as accuracy.",
    ]
    summary_text = "\n".join(summary_lines)
    (DIRS["final"] / "NEXT_STEP_AFTER_VISUAL_REVIEW.txt").write_text(
        summary_text,
        encoding="utf-8",
    )
    print(summary_text)


Patient: s11

This notebook completed signal-level screening only.

Previous best channels from config: ['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']
Top screening channels: ['F8', 'Fp1', 'F3', 'Ft7', 'Fp2']
Highest median predefined time window: T0_current_marker_0_1
Highest median predefined frequency band: F1_low_0p1_30

Before moving to model ablation:
1. Open 01_inventory_and_qc and exclude clearly unstable/artifact channels.
2. Inspect every per-channel ERP/trial card.
3. Inspect every per-channel TFR and discriminability card.
4. Check session stability; do not select a channel driven by one session only.
5. Edit channel_selection_sheet_EDIT_ME.csv.
6. Freeze selected/control/time/frequency candidates.
7. Run a separate fixed-fold SVM ablation; do not report these screening scores as accuracy.



## Критерий перехода к следующему ноутбуку

Переходить к fixed-fold SVM ablation можно после того, как для пациента вручную зафиксированы:

1. `selected_channels`;
2. `control_channels` того же размера;
3. небольшой неизменяемый список временных окон;
4. небольшой неизменяемый список частотных диапазонов;
5. причины исключения шумных или нестабильных каналов.

Следующий ноутбук должен использовать **одинаковые folds, preprocessing, epochs, seed и гиперпараметры SVM** для всех сравниваемых входов.
